In [140]:
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

sns.set_theme(style="whitegrid")

conn = sqlite3.connect("dsl.db")

from dsl_statistics.db import RANK_NAMES

# Division display order: Premier first, then numbered divisions
DIVISION_ORDER = ["Premier Division", "Division 1", "Division 2", "Division 3", "Division 4"]

def sorted_divisions(divisions):
    """Sort divisions in logical order (Premier first, then numbered)."""
    return sorted(divisions, key=lambda d: DIVISION_ORDER.index(d) if d in DIVISION_ORDER else len(DIVISION_ORDER))

In [141]:
# Get latest stats per player (most recent snapshot)
players_df = pd.read_sql("""
    SELECT
        p.id as player_id, p.display_name, p.steam_account_id,
        p.first_game_at, p.steam_account_created, p.steam_games_owned,
        p.steam_profile_visible,
        d.name as division, t.name as team,
        tm.role, tm.is_poc,
        ps.pp_score, ps.rank_number, ps.rank_subrank, ps.scraped_at
    FROM players p
    JOIN team_members tm ON p.id = tm.player_id AND tm.left_at IS NULL
    JOIN teams t ON tm.team_id = t.id
    JOIN divisions d ON t.division_id = d.id
    LEFT JOIN player_stats ps ON p.id = ps.player_id
        AND ps.scraped_at = (
            SELECT MAX(ps2.scraped_at) FROM player_stats ps2 WHERE ps2.player_id = p.id
        )
    ORDER BY d.name, t.name, p.display_name
""", conn)

players_df["rank_label"] = players_df["rank_number"].map(RANK_NAMES)
print(f"Loaded {len(players_df)} active players across {players_df['division'].nunique()} divisions")
players_df.head()

Loaded 801 active players across 5 divisions


,player_id,display_name,steam_account_id,first_game_at,steam_account_created,steam_games_owned,steam_profile_visible,division,team,role,is_poc,pp_score,rank_number,rank_subrank,scraped_at,rank_label
0,322,.twilley. (Jake),1120747002,None,2020-08-09T01:22:20+00:00,113.0,1,Division 1,100 IQ,core,1,6411.0,11,4,2026-04-18 07:06:06,Eternus
1,323,chikiaf,1127527955,None,2020-09-03T06:06:16+00:00,NaN,1,Division 1,100 IQ,core,0,5434.0,9,6,2026-04-18 07:06:11,Phantom
2,324,fruiwt (fruiwt),876989902,None,2018-05-22T22:32:46+00:00,124.0,1,Division 1,100 IQ,core,0,3286.0,6,2,2026-04-18 07:06:16,Emissary
3,325,jstn7. (mungy),416994670,None,NaN,NaN,0,Division 1,100 IQ,core,0,5743.0,10,3,2026-04-18 07:06:22,Ascendant
4,326,me_sushi (meesushii),1030495599,None,2019-09-14T23:00:41+00:00,36.0,1,Division 1,100 IQ,core,0,6707.0,11,6,2026-04-18 07:06:29,Eternus


In [142]:
div_stats = players_df.groupby("division")["pp_score"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).round(1)
div_stats.columns = ["Players", "Avg PP", "Median PP", "Std Dev", "Min PP", "Max PP"]
print("=== Division Overview ===")
div_stats

=== Division Overview ===


,Players,Avg PP,Median PP,Std Dev,Min PP,Max PP
division,,,,,,
Division 1,200,6005.7,6078.0,461.4,3286.0,6849.0
Division 2,203,5417.0,5477.0,584.3,3542.0,6716.0
Division 3,172,4525.2,4604.0,891.3,1133.0,6241.0
Division 4,108,3183.0,3213.0,1024.7,1034.0,5967.0
Premier Division,118,6293.3,6337.0,275.9,5602.0,7211.0


In [ ]:
# Average PP of core players per team
core_players = players_df[players_df["role"] == "core"]
team_rankings = core_players.groupby(["division", "team"])["pp_score"].agg(
    ["mean", "count"]
).round(1)
team_rankings.columns = ["Avg Core PP", "Core Players"]
team_rankings = team_rankings.sort_values(["Avg Core PP"], ascending=False)

for div in sorted_divisions(players_df["division"].unique()):
    div_teams = team_rankings.loc[div].sort_values("Avg Core PP", ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(4, len(div_teams) * 0.4)))
    div_teams["Avg Core PP"].plot(
        kind="barh", ax=ax, color=sns.color_palette("viridis", len(div_teams))
    )
    ax.set_title(f"{div} — Team Rankings by Avg Core PP")
    ax.set_xlabel("Average PP (Core Players)")
    plt.tight_layout()
    plt.show()

In [ ]:
team_pp = core_players.groupby(["division", "team"])["pp_score"].mean()

print("=== Outlier Teams (>1 std dev from division mean) ===\n")
for div in sorted_divisions(players_df["division"].unique()):
    div_team_pp = team_pp[div]
    mean = div_team_pp.mean()
    std = div_team_pp.std()
    outliers = div_team_pp[(div_team_pp > mean + std) | (div_team_pp < mean - std)]
    if len(outliers) > 0:
        print(f"{div} (mean={mean:.1f}, std={std:.1f}):")
        for team_name, pp in outliers.items():
            direction = "ABOVE" if pp > mean else "BELOW"
            print(f"  {team_name}: {pp:.1f} ({direction}, {abs(pp - mean)/std:.1f}σ)")
        print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PP histogram per division
for div in sorted_divisions(players_df["division"].unique()):
    div_data = players_df[players_df["division"] == div]["pp_score"].dropna()
    axes[0].hist(div_data, bins=20, alpha=0.5, label=div)
axes[0].set_title("PP Distribution by Division")
axes[0].set_xlabel("PP Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Rank distribution
rank_counts = players_df["rank_label"].value_counts()
rank_order = [RANK_NAMES[i] for i in range(12) if RANK_NAMES[i] in rank_counts.index]
rank_counts = rank_counts.reindex(rank_order)
rank_counts.plot(
    kind="bar", ax=axes[1], color=sns.color_palette("coolwarm", len(rank_counts))
)
axes[1].set_title("Rank Distribution (All Divisions)")
axes[1].set_xlabel("Rank")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
def scout_player(player_name: str):
    """Display scouting info for a player."""
    player = players_df[
        players_df["display_name"].str.contains(player_name, case=False)
    ]
    if player.empty:
        print(f"Player '{player_name}' not found")
        return

    p = player.iloc[0]
    print(f"=== {p['display_name']} ===")
    print(f"Team: {p['team']} ({p['division']})")
    print(f"Role: {p['role']}{'  [POC]' if p['is_poc'] else ''}")
    print(f"PP: {p['pp_score']:.1f}")
    print(f"Rank: {p['rank_label']} {p['rank_subrank']}")
    print(f"Steam Account Age: {p['steam_account_created'] or 'Unknown'}")
    print(f"Games Owned: {p['steam_games_owned'] or 'Unknown'}")
    print()

    # Top heroes
    heroes = pd.read_sql("""
        SELECT ph.hero_name, ph.matches_played, ph.win_rate
        FROM player_heroes ph
        JOIN player_stats ps ON ph.stats_id = ps.id
        WHERE ps.player_id = ?
        AND ps.scraped_at = (SELECT MAX(scraped_at) FROM player_stats WHERE player_id = ?)
        ORDER BY ph.matches_played DESC
        LIMIT 10
    """, conn, params=(int(p["player_id"]), int(p["player_id"])))

    if not heroes.empty:
        print("Top Heroes:")
        for _, h in heroes.iterrows():
            print(
                f"  {h['hero_name']}: {h['matches_played']} games, "
                f"{h['win_rate']*100:.0f}% WR"
            )

    # PP trend from match history
    matches = pd.read_sql("""
        SELECT match_date, pp_change, hero_name, result
        FROM player_matches
        WHERE player_id = ? AND pp_change IS NOT NULL
        ORDER BY match_date
    """, conn, params=(int(p["player_id"]),))

    if not matches.empty:
        # Reconstruct PP trajectory from current PP and cumulative changes
        matches["pp_estimated"] = p["pp_score"] - matches["pp_change"].iloc[::-1].cumsum().iloc[::-1] + matches["pp_change"]
        fig, ax = plt.subplots(figsize=(10, 4))
        colors = [
            "green" if r == "win" else "red" for r in matches["result"]
        ]
        ax.plot(
            range(len(matches)),
            matches["pp_estimated"],
            marker="o",
            color="steelblue",
        )
        ax.scatter(
            range(len(matches)),
            matches["pp_estimated"],
            c=colors,
            zorder=5,
        )
        ax.set_title(f"{p['display_name']} — Recent PP Trend")
        ax.set_xlabel("Match")
        ax.set_ylabel("PP")
        plt.tight_layout()
        plt.show()


# Example usage:
# scout_player("YourName")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
div_order = sorted_divisions(players_df["division"].unique())
team_avg_pp = core_players.groupby(["division", "team"])["pp_score"].mean()
data = [
    team_avg_pp[d].dropna() for d in div_order
]
colors = sns.color_palette("viridis", len(div_order))
bp = ax.boxplot(data, labels=div_order, patch_artist=True)
for i, color in enumerate(colors):
    bp["boxes"][i].set_facecolor(color)
    bp["boxes"][i].set_edgecolor("black")
    bp["whiskers"][2 * i].set_color("black")
    bp["whiskers"][2 * i + 1].set_color("black")
    bp["caps"][2 * i].set_color("black")
    bp["caps"][2 * i + 1].set_color("black")
    bp["medians"][i].set_color("black")
ax.set_title("Team Avg Core PP Distribution Across Divisions")
ax.set_ylabel("Average Core PP")
plt.tight_layout()
plt.show()

In [ ]:
# === Rebalancing Analysis (swap-based) ===
# For each adjacent pair of divisions (bottom up):
#   1. Find the highest outlier (>threshold σ above mean) in the lower division
#   2. If they're above the upper division's average, swap them with the
#      lowest team in the upper division
#   3. Recalculate division stats and repeat until no more valid swaps
# Swaps keep division sizes constant — Swiss format is automatically preserved.
# Premier is locked — promotion to Premier is earned by winning Div 1, not by PP.

# --- Tweak this ---
OUTLIER_THRESHOLD = 1.0  # σ above division mean to be considered for promotion
# ------------------

team_avg = core_players.groupby(["division", "team"])["pp_score"].mean().reset_index()
team_avg.columns = ["current_division", "team", "avg_pp"]
team_avg["proposed_division"] = team_avg["current_division"].copy()

def div_stats_for(div):
    pp = team_avg[team_avg["proposed_division"] == div]["avg_pp"]
    return pp.mean(), pp.std()

# Swappable boundaries: Div 4<->3, 3<->2, 2<->1 (skip Premier)
SWAPPABLE_DIVS = DIVISION_ORDER[1:]  # ["Division 1", "Division 2", "Division 3", "Division 4"]

swaps = []
max_iterations = 50
for iteration in range(max_iterations):
    swapped = False
    for i in range(len(SWAPPABLE_DIVS) - 1, 0, -1):
        lower_div = SWAPPABLE_DIVS[i]
        upper_div = SWAPPABLE_DIVS[i - 1]

        lower_teams = team_avg[team_avg["proposed_division"] == lower_div].sort_values("avg_pp", ascending=False)
        if len(lower_teams) < 2:
            continue

        lower_mean, lower_std = div_stats_for(lower_div)
        if lower_std == 0:
            continue

        # Find highest outlier in lower division
        top_team = lower_teams.iloc[0]
        sigma = (top_team["avg_pp"] - lower_mean) / lower_std
        if sigma <= OUTLIER_THRESHOLD:
            continue

        # Promote if team is above the upper division's average
        upper_mean, _ = div_stats_for(upper_div)
        if top_team["avg_pp"] < upper_mean:
            continue

        # Find the lowest team in the upper division to swap with
        upper_teams = team_avg[team_avg["proposed_division"] == upper_div].sort_values("avg_pp", ascending=True)
        bottom_team = upper_teams.iloc[0]

        # Only swap if promoted team is stronger than relegated team
        if top_team["avg_pp"] <= bottom_team["avg_pp"]:
            continue

        # Swap
        team_avg.loc[team_avg["team"] == top_team["team"], "proposed_division"] = upper_div
        team_avg.loc[team_avg["team"] == bottom_team["team"], "proposed_division"] = lower_div

        swaps.append({
            "promoted": top_team["team"],
            "promoted_pp": top_team["avg_pp"],
            "relegated": bottom_team["team"],
            "relegated_pp": bottom_team["avg_pp"],
            "upper_div": upper_div,
            "lower_div": lower_div,
            "sigma": sigma,
        })
        swapped = True

    if not swapped:
        break

print(f"Outlier threshold: {OUTLIER_THRESHOLD}σ | {len(swaps)} swaps in {iteration + 1} iterations")
print(f"(Premier Division locked — no swaps into/out of Premier)\n")

print(f"=== Proposed Swaps ===\n")
for s in swaps:
    print(f"  {s['lower_div']} <-> {s['upper_div']}:")
    print(f"    UP   {s['promoted']:40s}  {s['promoted_pp']:6.0f} PP  ({s['sigma']:.1f}σ above {s['lower_div']} mean)")
    print(f"    DOWN {s['relegated']:40s}  {s['relegated_pp']:6.0f} PP")
    print()

print(f"=== Current vs Proposed Division Summary ===\n")
print(f"{'Division':20s}  {'--- Current ---':>32s}  {'--- Proposed ---':>32s}")
for div in DIVISION_ORDER:
    cur = team_avg[team_avg["current_division"] == div]["avg_pp"]
    pro = team_avg[team_avg["proposed_division"] == div]["avg_pp"]
    cur_str = f"{len(cur):2d} teams  avg {cur.mean():5.0f}  std {cur.std():4.0f}"
    pro_str = f"{len(pro):2d} teams  avg {pro.mean():5.0f}  std {pro.std():4.0f}"
    lock = " (locked)" if div == "Premier Division" else ""
    print(f"{div:20s}  {cur_str:>32s}  {pro_str:>32s}{lock}")

# Side-by-side box plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = sns.color_palette("viridis", len(DIVISION_ORDER))

for ax, div_col, title in [
    (axes[0], "current_division", "Current Divisions"),
    (axes[1], "proposed_division", "Proposed (Swap-Based, Premier Locked)"),
]:
    data = [team_avg[team_avg[div_col] == d]["avg_pp"].dropna() for d in DIVISION_ORDER]
    bp = ax.boxplot(data, labels=DIVISION_ORDER, patch_artist=True)
    for i, color in enumerate(colors):
        bp["boxes"][i].set_facecolor(color)
        bp["boxes"][i].set_edgecolor("black")
        bp["whiskers"][2 * i].set_color("black")
        bp["whiskers"][2 * i + 1].set_color("black")
        bp["caps"][2 * i].set_color("black")
        bp["caps"][2 * i + 1].set_color("black")
        bp["medians"][i].set_color("black")
    ax.set_title(title)
    ax.set_ylabel("Avg Core PP")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# PP trajectory from match history for top players in a division
my_division = "Division 2"  # Change as needed

div_players = players_df[players_df["division"] == my_division].nlargest(
    5, "pp_score"
)

fig, ax = plt.subplots(figsize=(12, 5))
for _, p in div_players.iterrows():
    matches = pd.read_sql("""
        SELECT match_date, pp_change FROM player_matches
        WHERE player_id = ? AND pp_change IS NOT NULL
        ORDER BY match_date
    """, conn, params=(int(p["player_id"]),))
    if not matches.empty:
        # Reconstruct PP trajectory from current PP and cumulative changes (reverse)
        matches["pp_estimated"] = p["pp_score"] - matches["pp_change"].iloc[::-1].cumsum().iloc[::-1] + matches["pp_change"]
        ax.plot(
            range(len(matches)),
            matches["pp_estimated"],
            label=p["display_name"],
            marker=".",
        )

ax.set_title(f"PP Trends — Top 5 Players in {my_division}")
ax.set_xlabel("Match Index")
ax.set_ylabel("PP")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
alt_flags = players_df[
    (players_df["pp_score"].notna())
    & (
        (players_df["steam_profile_visible"] == False)
        | (
            players_df["steam_games_owned"].notna()
            & (players_df["steam_games_owned"] < 10)
        )
        | (
            players_df["steam_account_created"].notna()
            & (
                pd.to_datetime(players_df["steam_account_created"], utc=True)
                > pd.Timestamp("2024-06-01", tz="UTC")
            )
        )
    )
].sort_values("pp_score", ascending=False)

print(f"=== Potential Alt Accounts ({len(alt_flags)} flagged) ===\n")
for _, p in alt_flags.iterrows():
    flags = []
    if p["steam_profile_visible"] == False:
        flags.append("PRIVATE PROFILE")
    if pd.notna(p["steam_games_owned"]) and p["steam_games_owned"] < 10:
        flags.append(f"ONLY {int(p['steam_games_owned'])} GAMES")
    if pd.notna(p["steam_account_created"]):
        created = pd.to_datetime(p["steam_account_created"], utc=True)
        if created > pd.Timestamp("2024-06-01", tz="UTC"):
            flags.append(f"ACCOUNT CREATED {created.strftime('%Y-%m')}")

    print(f"  {p['display_name']} ({p['team']}, {p['division']})")
    print(
        f"    PP: {p['pp_score']:.1f} | Rank: {p['rank_label']} | "
        f"Flags: {', '.join(flags)}"
    )

In [ ]:
conn.close()